In [1]:
# ── Parameters ─────────────────────────────────────────────────────────────────────────────────────────────
PIPELINE_RUN_ID = ''
EXAMPLE_POS     = ['13219558', '13219560', '13219566', '13219607', '13219824', '13219834', '13148876', '4508252311']
CREATED_FROM    = '2026-09-01'      # the business extract: POs created on or after this date (EKKO AEDAT)
SOURCE_ABFSS    = ''                # optional: abfss path of the DAB lakehouse that feeds PROC Bronze, to compare EKET / EKBE at the source
SOURCE_SCHEMA   = 'dbo'
N_EXAMPLES      = 20

StatementMeta(, 30179317-6ee6-46c7-828b-20981f14cc60, 3, Finished, Available, Finished, False)

In [2]:
%run ./NB_Utils_Common

StatementMeta(, 30179317-6ee6-46c7-828b-20981f14cc60, 9, Finished, Available, Finished, True)

NB_Utils_Common: settings ready
NB_Utils_Common: locations ready
NB_Utils_Common: start_run() ready
NB_Utils_Common: OPS logging ready
NB_Utils_Common: dq_record(), dq_flush(), check_volume_change() ready
NB_Utils_Common loaded: start_run(), tpath(), record_table(), dq_record(), end_of_run_report(), end_session()


In [3]:
%run ./NB_Config_PROC

StatementMeta(, 30179317-6ee6-46c7-828b-20981f14cc60, 16, Finished, Available, Finished, True)

Config loaded: ['EKKO', 'EKPO', 'EKET', 'EKBE']
EA audit report tables added: EKES, EKKN, EKPA, EKAB
New tables added:EKPV, EBKN, EBAN


In [4]:
%run ./NB_Utils_Gold

StatementMeta(, 30179317-6ee6-46c7-828b-20981f14cc60, 17, Finished, Available, Finished, True)

NB_Utils_Gold loaded: silver(), gold(), drop_redundant(), sk_lookup(), write_gold(), stable_sk(), promote_to_gold(), profile_strings()


In [5]:
# ── Setup ──────────────────────────────────────────────────────────────────────────────────────────────────
start_run('PROC_Check_Feedback_V4', 'CHECK')
fact  = gold('Fact_PO_LineItem_Schedule')
ev    = gold('Fact_PO_Order_Status_Evidence')
eket, ekpo, ekko, ekbe, ekes = silver('EKET'), silver('EKPO'), silver('EKKO'), silver('EKBE'), silver('EKES')
_nz   = lambda c: F.regexp_replace(F.col(c).cast('string'), '^0+', '')
# the POs of the extract: created on or after CREATED_FROM
pos = ekko.filter(F.col('Created_On') >= F.lit(CREATED_FROM)).select('Purchasing_Document', 'Created_On', 'Purch_Doc_Category')
# goods receipts per PO item (physical GR: VGABE 1 / BEWTP E), as cell 20 of NB_Gold_PROC counts them
gr_item = (ekbe.filter(F.col('Transevent_Type').eqNullSafe('1') | F.col('PO_History_Category').eqNullSafe('E'))
           .groupBy('Purchasing_Document', 'Item')
           .agg(F.sum(F.col('Signed_Quantity').cast('double')).alias('Item_Gr_Qty'), F.max('Posting_Date').alias('Last_GR_Date'),
                F.count('*').alias('GR_Rows')))
for _n, _df in [('fact', fact), ('ev', ev), ('eket', eket), ('ekpo', ekpo), ('ekko', ekko), ('ekbe', ekbe), ('ekes', ekes),
                ('pos', pos), ('gr_item', gr_item)]:
    _df.createOrReplaceTempView(_n)
print(f"Fact_PO_LineItem_Schedule: {fact.count():,} rows | POs created since {CREATED_FROM}: {pos.count():,}")
_ex = ', '.join(f"'{p}'" for p in EXAMPLE_POS)

StatementMeta(, 30179317-6ee6-46c7-828b-20981f14cc60, 18, Finished, Available, Finished, False)

Pipeline : PROC_Check_Feedback_V4
Run ID   : 1885188c-ea4a-4f8b-bb1c-e47c151dfd95
Started  : 2026-11-19 20:49:12 UTC
Locations:
  bronze     dbo              abfss://3532debe-0172-4493-815c-dc7aed901ac7@onelake.dfs.fabric.micros…
  silver     dbo              abfss://3532debe-0172-4493-815c-dc7aed901ac7@onelake.dfs.fabric.micros…
  gold       dbo              abfss://3532debe-0172-4493-815c-dc7aed901ac7@onelake.dfs.fabric.micros…
  md_silver  dbo              abfss://9cb5a2e3-2121-4f0d-8289-c1f350ac646f@onelake.dfs.fabric.micros…
  md_gold    dbo              abfss://9cb5a2e3-2121-4f0d-8289-c1f350ac646f@onelake.dfs.fabric.micros…
  ops        dbo              abfss://548c95f1-2c7c-44e4-a26f-e6971f80339f@onelake.dfs.fabric.micros…
  sd_silver  dbo              abfss://303715d4-7108-4456-a1f2-bbd04c28cc0c@onelake.dfs.fabric.micros…
  xrate      Exchange_Rates   abfss://9cb5a2e3-2121-4f0d-8289-c1f350ac646f@onelake.dfs.fabric.micros…
Fact_PO_LineItem_Schedule: 6,914,717 rows | POs created 

### Q1 · D1 — lines with no goods receipt in Fabric
**Issue:** 85,777 lines are Open in Fabric and Closed by GR in Every Angle; all have EKET WEMNG 0 and no EKBE row. **Expected:** about the same number here if Fabric lacks the receipts. **Feeds:** whether Amarildo and team must fix the EKET / EKBE extract.

In [6]:
# ── Q1 · D1: schedule lines with no goods receipt in Fabric, by month and plant ────────────────────────────────
# Every Angle closes them by GR. Expected: about 86,000 lines (the extract) when Fabric lacks the receipts; far fewer when the
# extract was taken from an old run and Fabric has caught up since.
q1 = spark.sql(f'''
SELECT date_format(p.Created_On, 'yyyy-MM') AS created_month, i.Plant,
       SUM(CASE WHEN COALESCE(l.Quantity_Delivered, 0) = 0 AND g.Item_Gr_Qty IS NULL AND COALESCE(l.Scheduled_Quantity, 0) > 0 THEN 1 ELSE 0 END) AS lines_no_receipt,
       SUM(CASE WHEN COALESCE(l.Quantity_Delivered, 0) > 0 OR g.Item_Gr_Qty IS NOT NULL THEN 1 ELSE 0 END)   AS lines_with_receipt,
       COUNT(*) AS lines
FROM eket l JOIN pos p ON p.Purchasing_Document = l.Purchasing_Document
LEFT JOIN ekpo i ON i.Purchasing_Document = l.Purchasing_Document AND i.Item = l.Item
LEFT JOIN gr_item g ON g.Purchasing_Document = l.Purchasing_Document AND g.Item = l.Item
GROUP BY 1, 2 ORDER BY 1, 2''')
display(q1)
_tot = q1.agg(F.sum('lines_no_receipt'), F.sum('lines_with_receipt')).first()
print(f'Lines without any receipt in Fabric: {_tot[0] or 0:,} | with a receipt: {_tot[1] or 0:,}')

StatementMeta(, 30179317-6ee6-46c7-828b-20981f14cc60, 19, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 00e7c510-96f3-4404-9822-61f25b863e81)

Lines without any receipt in Fabric: 65,502 | with a receipt: 236,991


### Q2 · D1 / D10 — Bronze freshness
**Issue:** goods receipts update existing EKET rows and add EKBE rows. **Expected:** recent `_UPDATED_AT`, `rows_changed > 0` for EKET. **Feeds:** extract diagnosis.

In [7]:
# ── Q2 · D1 / D10: are Bronze EKKO, EKET and EKBE still being refreshed? ──────────────────────────────────────
# Expected: _UPDATED_AT close to the last run for all three; rows_changed > 0 for EKET on recent runs (goods receipts update
# WEMNG on existing EKET rows). A table whose rows_changed is always 0 only receives inserts.
for _t in ['EKKO', 'EKET', 'EKBE']:
    _b = read_table('bronze', _t)
    _r = _b.agg(F.count('*'), F.max('_UPDATED_AT'), F.sum(F.col('_IS_DELETED').cast('int'))).first()
    print(f'Bronze {_t}: {_r[0]:,} rows, last _UPDATED_AT {_r[1]}, deleted {_r[2] or 0:,}')
    try:
        display(spark.sql(f"DESCRIBE HISTORY delta.`{tpath('bronze', _t)}`").select('version', 'timestamp', 'operation', 'operationMetrics').limit(5))
    except Exception as e:
        print(f'  history not readable ({str(e)[:100]})')
try:
    _log = read_table('ops', 'table_log')
    display(_log.filter(F.col('table_name').isin('EKKO', 'EKET', 'EKBE') & (F.col('logged_at') >= F.lit('2026-11-01')))
                .select('logged_at', 'layer', 'table_name', 'status', 'load_mode', 'rows_source', 'rows_in', 'rows_new', 'rows_changed',
                        'rows_deleted', 'rows_out', 'duration_s')
                .orderBy(F.desc('logged_at')))
except Exception as e:
    print(f'OPS table_log not readable ({str(e)[:120]})')

StatementMeta(, 30179317-6ee6-46c7-828b-20981f14cc60, 20, Finished, Available, Finished, False)

Bronze EKKO: 3,401,432 rows, last _UPDATED_AT 2026-11-19 08:18:15.750734, deleted 0


SynapseWidget(Synapse.DataFrame, eaa17f2f-ff96-4b18-bac6-393386ed9b00)

Bronze EKET: 6,914,732 rows, last _UPDATED_AT 2026-11-19 08:15:36.154512, deleted 0


SynapseWidget(Synapse.DataFrame, 8069767f-9376-4a64-8310-23364e431b33)

Bronze EKBE: 17,889,092 rows, last _UPDATED_AT 2026-11-19 08:18:29.050113, deleted 0


SynapseWidget(Synapse.DataFrame, 609dc4d3-2732-437c-9379-ce45ccc01af8)

SynapseWidget(Synapse.DataFrame, 22c76235-40c0-4407-a67a-1430dcff2ed9)

### Q3 · D1 — example POs through every layer
**Issue:** where does the receipt disappear. **Expected:** Bronze already lacks it (source gap). **Feeds:** who fixes it.

In [8]:
# ── Q3 · D1: the example POs through every layer ─────────────────────────────────────────────────────────────
# Expected (if D1 is a source gap): EKET WEMNG 0 and no EKBE rows in Bronze already. If Bronze has the receipt and Silver or
# Gold does not, the loss is ours (then see Q4).
_br_eket = read_table('bronze', 'EKET').filter(_nz('EBELN').isin([p.lstrip('0') for p in EXAMPLE_POS]))
display(_br_eket.select('EBELN', 'EBELP', 'ETENR', 'EINDT', 'MENGE', 'WEMNG', 'DABMG', 'MNG02', '_UPDATED_AT', '_IS_DELETED').orderBy('EBELN', 'EBELP', 'ETENR'))
_br_ekbe = read_table('bronze', 'EKBE').filter(_nz('EBELN').isin([p.lstrip('0') for p in EXAMPLE_POS]))
print(f'Bronze EKBE rows for the examples: {_br_ekbe.count():,}')
display(_br_ekbe.select('EBELN', 'EBELP', 'VGABE', 'BEWTP', 'BWART', 'BUDAT', 'MENGE', 'SHKZG', 'ETENS', '_UPDATED_AT').orderBy('EBELN', 'EBELP', 'BUDAT'))
display(spark.sql(f'''
SELECT l.Purchasing_Document, l.Item, l.Schedule_Line, l.Scheduled_Quantity, l.Quantity_Delivered, l.Qty_Reduced_MRP, l.Committed_Quantity,
       i.Delivery_Completed, g.Item_Gr_Qty, g.GR_Rows, g.Last_GR_Date
FROM eket l LEFT JOIN ekpo i ON i.Purchasing_Document = l.Purchasing_Document AND i.Item = l.Item
LEFT JOIN gr_item g ON g.Purchasing_Document = l.Purchasing_Document AND g.Item = l.Item
WHERE l.Purchasing_Document IN ({_ex}) ORDER BY 1, 2, 3'''))
display(fact.filter(F.col('Purchasing_Document').isin(EXAMPLE_POS))
            .select('Purchasing_Document', 'Item', 'Schedule_Line', 'Order_Status_Purchase', 'Execution_Status', 'Scheduled_Quantity',
                    'Quantity_Delivered', 'Realized_Finish_Date', 'Piece_Count', 'Confirmation_Categories')
            .orderBy('Purchasing_Document', 'Item', 'Schedule_Line'))
if SOURCE_ABFSS:
    for _t, _cols in [('EKET', ['EBELN', 'EBELP', 'ETENR', 'MENGE', 'WEMNG', 'DABMG']), ('EKBE', ['EBELN', 'EBELP', 'VGABE', 'BWART', 'BUDAT', 'MENGE'])]:
        try:
            _s = spark.read.format('delta').load(f'{SOURCE_ABFSS}/Tables/{SOURCE_SCHEMA}/{_t}')
            display(_s.filter(_nz('EBELN').isin([p.lstrip('0') for p in EXAMPLE_POS])).select(*_cols))
        except Exception as e:
            print(f'Source {_t} not readable ({str(e)[:120]})')
else:
    print('SOURCE_ABFSS not set: the DAB source is not compared (Amarildo and team can run the same selection there)')

StatementMeta(, 30179317-6ee6-46c7-828b-20981f14cc60, 21, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 7463c5ed-545b-4c8a-a14f-adb1b0579409)

Bronze EKBE rows for the examples: 29


SynapseWidget(Synapse.DataFrame, f914d491-3c0a-44c1-af1d-bcf5ab0b2824)

SynapseWidget(Synapse.DataFrame, 8c19c500-ab17-4fbc-b409-1b9941d64b3f)

SynapseWidget(Synapse.DataFrame, aefeca1f-704d-4319-9d20-919543ec6af1)

SOURCE_ABFSS not set: the DAB source is not compared (Amarildo and team can run the same selection there)


### Q4 · D1 — EKBE keys
**Issue:** a leading-zero or type mismatch would drop receipts in Silver / Gold. **Expected:** 0 unmatched, one key length. **Feeds:** a config fix only if this fails.

In [9]:
# ── Q4 · D1: do the EKBE keys match EKPO? ────────────────────────────────────────────────────────────────────
# Expected: 0 unmatched rows and one key length per table. Unmatched rows or two lengths mean a leading-zero / type problem.
display(spark.sql('''
SELECT COUNT(*) AS ekbe_rows,
       SUM(CASE WHEN i.Purchasing_Document IS NULL THEN 1 ELSE 0 END) AS no_ekpo_match
FROM ekbe b LEFT JOIN ekpo i ON i.Purchasing_Document = b.Purchasing_Document AND i.Item = b.Item'''))
for _n in ['ekbe', 'eket', 'ekpo']:
    display(spark.sql(f'''SELECT '{_n}' AS tbl, LENGTH(Purchasing_Document) AS po_len, LENGTH(Item) AS item_len, COUNT(*) AS rows
                          FROM {_n} GROUP BY 1, 2, 3 ORDER BY 4 DESC'''))

StatementMeta(, 30179317-6ee6-46c7-828b-20981f14cc60, 22, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, b9bd547b-4d06-4713-a397-b2fd605e5b7d)

SynapseWidget(Synapse.DataFrame, 79b76d40-f980-4b75-8842-4e7b96824da2)

SynapseWidget(Synapse.DataFrame, 020303e1-aef9-41ad-bb58-4df41e1be207)

SynapseWidget(Synapse.DataFrame, 65279c39-14e7-4db7-a8f6-347d461e2429)

### Q5 · D3 — zero-quantity schedule lines
**Issue:** 27,392 items end with a line of quantity 0 that Every Angle hides; 7,940 of those lines show a receipt. **Expected:** the committed quantity equals the item's scheduled quantity; receipts on zero lines explained. **Feeds:** business decision D3.

In [10]:
# ── Q5 · D3: zero-quantity schedule lines ────────────────────────────────────────────────────────────────────
# Expected: on most items the zero line is the last one and its committed quantity equals the other lines' scheduled quantity
# (an EKET row carrying the ATP commitment; Every Angle hides it). A zero line with WEMNG > 0 should not exist: list them.
_zl = spark.sql('''
WITH l AS (SELECT Purchasing_Document, Item, Schedule_Line, COALESCE(Scheduled_Quantity, 0) AS sched, COALESCE(Quantity_Delivered, 0) AS recv,
                  COALESCE(Committed_Quantity, 0) AS com, MAX(Schedule_Line) OVER (PARTITION BY Purchasing_Document, Item) AS last_line,
                  SUM(COALESCE(Scheduled_Quantity, 0)) OVER (PARTITION BY Purchasing_Document, Item) AS item_sched
           FROM eket)
SELECT COUNT(*) AS zero_lines,
       SUM(CASE WHEN Schedule_Line = last_line THEN 1 ELSE 0 END)            AS zero_is_last_line,
       SUM(CASE WHEN ABS(com - item_sched) < 0.001 THEN 1 ELSE 0 END)        AS committed_equals_item_scheduled,
       SUM(CASE WHEN recv > 0 THEN 1 ELSE 0 END)                             AS zero_lines_with_receipt
FROM l WHERE sched = 0 AND com > 0''')
display(_zl)
display(spark.sql(f'''
SELECT l.Purchasing_Document, l.Item, l.Schedule_Line, l.Scheduled_Quantity, l.Quantity_Delivered, l.Committed_Quantity, l.Committed_Date,
       l.Delivery_Date, g.Item_Gr_Qty, g.GR_Rows
FROM eket l LEFT JOIN gr_item g ON g.Purchasing_Document = l.Purchasing_Document AND g.Item = l.Item
WHERE COALESCE(l.Scheduled_Quantity, 0) = 0 AND COALESCE(l.Quantity_Delivered, 0) > 0
ORDER BY l.Purchasing_Document, l.Item LIMIT {N_EXAMPLES}'''))

StatementMeta(, 30179317-6ee6-46c7-828b-20981f14cc60, 23, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 03905a34-c12b-4ece-8ebd-feb7b4d7297f)

SynapseWidget(Synapse.DataFrame, a140189c-d178-4321-a17d-bca7c10bfd31)

### Q6 · D5 — Closed by GR vs Outbound delivery created
**Issue:** 8,439 fully received stock-transfer lines. **Expected:** PSTYP U with a delivery. **Feeds:** rule order for stock transfers (business).

In [11]:
# ── Q6 · D5: Closed by GR lines on stock transfers (Every Angle: Outbound delivery created) ───────────────────
# Expected: PSTYP U items, fully received, with an outbound delivery (LIPS) or an inbound delivery on the confirmation. The
# question for the business is the rule order for these lines (rule 2 Closed by GR before rule 10 today).
_sto = (fact.filter(F.col('Order_Status_Purchase') == 'Closed by GR')
            .join(ekpo.select('Purchasing_Document', 'Item', 'Item_Category_Ext'), ['Purchasing_Document', 'Item'], 'left')   # Delivery_Completed is on the fact already (18 Nov 2026)
            .filter(F.trim(F.col('Item_Category_Ext')).eqNullSafe('U')))
print(f"Closed by GR on stock-transfer items (PSTYP U): {_sto.count():,} lines")
_mv = ekbe.groupBy('Purchasing_Document', 'Item').agg(F.concat_ws('|', F.sort_array(F.collect_set('Movement_Type'))).alias('Movement_Types'))
_ib = ekes.groupBy('Purchasing_Document', 'Item').agg(F.max(F.when(F.trim(F.coalesce(F.col('Inbound_Delivery'), F.lit(''))) != '', F.col('Inbound_Delivery'))).alias('Inbound_Delivery'))
_q6 = (_sto.join(_mv, ['Purchasing_Document', 'Item'], 'left').join(_ib, ['Purchasing_Document', 'Item'], 'left'))
if 'sd_silver' in LOC and table_exists('sd_silver', 'LIPS'):
    _lips = (read_table('sd_silver', 'LIPS', active=True).select(_nz('Reference_Document').alias('Purchasing_Document'), _nz('Reference_Item').alias('Item'))
             .dropDuplicates().withColumn('Has_LIPS_Delivery', F.lit(True)))
    _q6 = _q6.join(_lips, ['Purchasing_Document', 'Item'], 'left')
else:
    print('SD Silver LIPS not reachable: Has_LIPS_Delivery not shown')
    _q6 = _q6.withColumn('Has_LIPS_Delivery', F.lit(None).cast('boolean'))
display(_q6.select('Purchasing_Document', 'Item', 'Schedule_Line', 'Execution_Status', 'Scheduled_Quantity', 'Quantity_Delivered',
                   'Qty_Delivered_STO', 'Committed_Quantity', 'Delivery_Completed', 'Movement_Types', 'Inbound_Delivery', 'Has_LIPS_Delivery')
           .orderBy('Purchasing_Document', 'Item', 'Schedule_Line').limit(N_EXAMPLES))

StatementMeta(, 30179317-6ee6-46c7-828b-20981f14cc60, 24, Finished, Available, Finished, False)

Closed by GR on stock-transfer items (PSTYP U): 2,174,989 lines


SynapseWidget(Synapse.DataFrame, cd522e70-ef80-46fa-923f-a28e734b4d83)

### Q7 · D6 — open-line status pairs
**Issue:** four pairs where both sides see an open line but name it differently. **Expected:** the EKES rows and committed quantities show which evidence Every Angle used. **Feeds:** business decisions on rules 10, 11, 14, 15.

In [12]:
# ── Q7 · D6: the four open-line status pairs — what evidence each side sees ─────────────────────────────────
# Fabric → Every Angle: To be confirmed → Stock committed (2,003 in the extract); Stock committed → Outbound delivery created
# (1,081); Shipping notification → Confirmed (945); To be confirmed → Outbound delivery created (1,166). For N_EXAMPLES lines of
# each Fabric status: the item's EKES rows (category, dates, quantities), committed quantity and date, PSTYP, BSTAE / KZABS / LABNR.
_conf = (ekes.filter(F.trim(F.coalesce(F.col('Confirmation_Deletion'), F.lit(''))) == '')
             .groupBy('Purchasing_Document', 'Item')
             .agg(F.count('*').alias('EKES_Rows'),
                  F.concat_ws('|', F.sort_array(F.collect_set('Confirmation_Category'))).alias('EKES_Categories'),
                  F.max('Confirmed_Delivery_Date').alias('Last_Confirmed_Date'), F.max('Created_On').alias('Last_Conf_Created'),
                  F.sum(F.col('Confirmation_Quantity').cast('double')).alias('Conf_Qty'),
                  F.sum(F.col('Confirmation_Qty_Reduced_MRP').cast('double')).alias('Conf_Qty_Reduced'),
                  F.max(F.when(F.trim(F.coalesce(F.col('Inbound_Delivery'), F.lit(''))) != '', F.col('Inbound_Delivery'))).alias('Inbound_Delivery')))
_itm = ekpo.select('Purchasing_Document', 'Item', 'Item_Category_Ext', 'Confirmation_Control', 'Order_Acknowledgment')   # Acknowledgment_Reqd is on the fact already (18 Nov 2026)
for _status in ['To be confirmed', 'Stock committed', 'Shipping notification']:
    print(f'\n{_status}: open lines, {N_EXAMPLES} examples')
    display(fact.filter((F.col('Order_Status_Purchase') == _status) & (F.coalesce(F.col('Quantity_Delivered'), F.lit(0)) == 0))
                .join(_conf, ['Purchasing_Document', 'Item'], 'left').join(_itm, ['Purchasing_Document', 'Item'], 'left')
                .select('Purchasing_Document', 'Item', 'Schedule_Line', 'Delivery_Date', 'Scheduled_Quantity', 'Committed_Quantity', 'Committed_Date',
                        'Item_Category_Ext', 'Confirmation_Control', 'Acknowledgment_Reqd', 'Order_Acknowledgment',
                        'EKES_Rows', 'EKES_Categories', 'Last_Confirmed_Date', 'Last_Conf_Created', 'Conf_Qty', 'Conf_Qty_Reduced', 'Inbound_Delivery',
                        'Piece_Count', 'Confirmation_Categories')
                .orderBy('Purchasing_Document', 'Item', 'Schedule_Line').limit(N_EXAMPLES))

StatementMeta(, 30179317-6ee6-46c7-828b-20981f14cc60, 25, Finished, Available, Finished, False)


To be confirmed: open lines, 20 examples


SynapseWidget(Synapse.DataFrame, 6eb60b26-2d83-4a0f-8acb-092ce928b0f5)


Stock committed: open lines, 20 examples


SynapseWidget(Synapse.DataFrame, f204213a-7464-4a1b-ba12-7bf393a1037d)


Shipping notification: open lines, 20 examples


SynapseWidget(Synapse.DataFrame, 8b53a6bf-c351-4ab9-8dae-3b1587b38ad5)

### Q8 · D10 — missing PO
**Issue:** PO 13148876 (created 24 Sep) is in Every Angle, not in Fabric. **Expected:** absent from Bronze or flagged deleted. **Feeds:** Amarildo and team.

In [13]:
# ── Q8 · D10: the PO Every Angle has and Fabric lacks ───────────────────────────────────────────────────────
# Expected: no row in Bronze EKKO (extract gap) or a row with _IS_DELETED = true (deleted upstream). The business should send
# the full list of missing POs; EXAMPLE_POS has the one example from the file (13148876).
for _t, _key in [('EKKO', 'EBELN'), ('EKPO', 'EBELN'), ('EKET', 'EBELN')]:
    _b = read_table('bronze', _t).filter(_nz(_key).isin([p.lstrip('0') for p in EXAMPLE_POS]))
    display(_b.groupBy(_key, '_IS_DELETED').agg(F.count('*').alias('rows'), F.min('_CREATED_AT').alias('first_loaded'), F.max('_UPDATED_AT').alias('last_updated'))
              .withColumn('table', F.lit(_t)).orderBy(_key))

StatementMeta(, 30179317-6ee6-46c7-828b-20981f14cc60, 26, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 72523066-c0d8-497b-b63f-68d39ed9b3dc)

SynapseWidget(Synapse.DataFrame, 3222db50-7179-4aa4-a41c-445872f1aec6)

SynapseWidget(Synapse.DataFrame, 00cf5373-0271-458a-ade8-9cfe91ba0575)

### Q9 · D7 / D8 — duplicates and over-confirmed pieces
**Expected after the 17 Nov change:** 0 keys with more than one row, 0 rows without a schedule line; the pieces are in the evidence table.

In [14]:
# ── Q9 · D7 / D8: duplicated keys and over-confirmed pieces ─────────────────────────────────────────────────
# Before 17 Nov 2026 the fact was one row per segment: a partly confirmed line had two rows (259 keys in the extract) and the
# over-confirmed pieces had no schedule line (4,398 rows). Expected now: 0 and 0 in the fact; the pieces are in the evidence table.
_dup = fact.groupBy('Purchasing_Document', 'Item', 'Schedule_Line').count().filter('count > 1')
print(f"Keys with more than one row in the fact: {_dup.count():,} (expected 0) | rows with an empty Schedule_Line: "
      f"{fact.filter(F.col('Schedule_Line').isNull()).count():,} (expected 0)")
print(f"Evidence table: {ev.count():,} segments; over-confirmed pieces (no schedule line): "
      f"{ev.filter(F.coalesce(F.col('Is_Over_Confirmed'), F.lit(False))).count():,}; lines with more than one piece: "
      f"{fact.filter(F.col('Piece_Count') > 1).count():,}")
display(fact.filter(F.col('Piece_Count') > 1)
            .select('Purchasing_Document', 'Item', 'Schedule_Line', 'Order_Status_Purchase', 'Piece_Count', 'Confirmation_Categories', 'Segment_Quantity')
            .orderBy('Purchasing_Document', 'Item', 'Schedule_Line').limit(N_EXAMPLES))

StatementMeta(, 30179317-6ee6-46c7-828b-20981f14cc60, 27, Finished, Available, Finished, False)

Keys with more than one row in the fact: 0 (expected 0) | rows with an empty Schedule_Line: 0 (expected 0)
Evidence table: 6,934,450 segments; over-confirmed pieces (no schedule line): 18,414; lines with more than one piece: 631


SynapseWidget(Synapse.DataFrame, 701619f1-96a4-4cbf-b7a6-3dcff2b62f40)

### Q10 · D2 — line-level execution status
**Issue:** 9,292 lines fully received show Partially Open because the status is the item's. **Expected:** those lines become Closed. **Feeds:** business decision D2.

In [15]:
# ── Q10 · D2: a line-level execution status against the item-level one ───────────────────────────────────────
# Proposal D2 (NB_Gold_PROC cell 22, commented): Cancelled when the item is; Closed when ELIKZ is set or nothing is left to receive;
# Partially Open when some is received; else Open. Expected: the Partially Open → Closed pairs (9,292 in the extract) move.
_recv, _sched = F.coalesce(F.col('Quantity_Delivered'), F.lit(0)), F.coalesce(F.col('Scheduled_Quantity'), F.lit(0))
_line_status = (F.when(F.col('Execution_Status').isin('Cancelled_Header', 'Cancelled_Line'), F.col('Execution_Status'))
                 .when((F.col('Delivery_Completed') == 'X') | ((_sched - _recv) <= OS_TOLERANCE), 'Closed')
                 .when(_recv > 0, 'Partially Open').otherwise('Open'))
_q10 = fact.withColumn('Line_Execution_Status', _line_status)
display(_q10.groupBy('Execution_Status', 'Line_Execution_Status').count().orderBy(F.desc('count')))
print(f"Lines whose status would change: {_q10.filter(F.col('Execution_Status') != F.col('Line_Execution_Status')).count():,} of {fact.count():,}")

StatementMeta(, 30179317-6ee6-46c7-828b-20981f14cc60, 28, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 06f2a9ea-b967-479c-99e8-54104449b454)

Lines whose status would change: 8,388 of 6,914,717


In [16]:
# ── End-of-run report: result, issues in plain language, tables, checks ────────
end_of_run_report()

StatementMeta(, 30179317-6ee6-46c7-828b-20981f14cc60, 29, Finished, Available, Finished, True)

════════════════════════════════════════════════════════════════════════════════════════════════════
 PROC_Check_Feedback_V4  |  run 1885188c-ea4a-4f8b-bb1c-e47c151dfd95
 19 Nov 2026 20:49–20:54 UTC (5m 24s)  |  0 of 0 tables OK  |  0 checks
 RESULT: ✓ All 0 tables loaded, all checks passed
════════════════════════════════════════════════════════════════════════════════════════════════════
 Details: SELECT * FROM dq_results WHERE pipeline_run_id = '1885188c-ea4a-4f8b-bb1c-e47c151dfd95' ORDER BY status
════════════════════════════════════════════════════════════════════════════════════════════════════
Session stopped to release capacity (END_SESSION = False keeps it)
  Session not stopped (__$$_STOP_SESSION_REQUEST_$$__)


'SUCCEEDED'